# GeoLifeCLEF 2025 — XGBoost + BioclimTimeSeries
**Amélioration** : Features tabulaires + statistiques des séries temporelles climatiques

Baseline précédente : **0.16395** → Objectif : **0.20+**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

DATA = Path('/kaggle/input/competitions/le-challenge-deep-learning-miashs-2026')
ENV  = DATA / 'EnvironmentalValues'
print('OK')

In [ ]:
# ─── 1. PA metadata ───────────────────────────────────────────────────────────
pa_train = pd.read_csv(DATA / 'GLC25_PA_metadata_train.csv')
pa_test  = pd.read_csv(DATA / 'GLC25_PA_metadata_test.csv')

train_surveys = pa_train[['surveyId','lon','lat','year','region']].drop_duplicates('surveyId')
test_surveys  = pa_test[['surveyId','lon','lat','year','region']].drop_duplicates('surveyId')

train_labels = (
    pa_train.dropna(subset=['speciesId'])
    .groupby('surveyId')['speciesId']
    .apply(lambda x: list(x.astype(int).unique()))
    .reset_index()
    .rename(columns={'speciesId': 'species_list'})
)
print(f'Train: {len(train_surveys)} | Test: {len(test_surveys)} | Espèces: {pa_train["speciesId"].nunique()}')

In [ ]:
# ─── 2. Features environnementales tabulaires ─────────────────────────────────
def load_env(tr, te):
    return pd.read_csv(tr), pd.read_csv(te)

elev_tr, elev_te       = load_env(ENV/'Elevation/GLC25-PA-train-elevation.csv',
                                   ENV/'Elevation/GLC25-PA-test-elevation.csv')
soil_tr, soil_te       = load_env(ENV/'SoilGrids/GLC25-PA-train-soilgrids.csv',
                                   ENV/'SoilGrids/GLC25-PA-test-soilgrids.csv')
land_tr, land_te       = load_env(ENV/'LandCover/GLC25-PA-train-landcover.csv',
                                   ENV/'LandCover/GLC25-PA-test-landcover.csv')
foot_tr, foot_te       = load_env(ENV/'HumanFootprint/GLC25-PA-train-human_footprint.csv',
                                   ENV/'HumanFootprint/GLC25-PA-test-human_footprint.csv')
bioclim_tr, bioclim_te = load_env(ENV/'ClimateAverage_1981-2010/GLC25-PA-train-bioclimatic.csv',
                                   ENV/'ClimateAverage_1981-2010/GLC25-PA-test-bioclimatic.csv')
print('Features tabulaires chargées')

In [ ]:
# ─── 3. BioclimTimeSeries → stats résumées ────────────────────────────────────
print('Chargement BioclimTimeSeries...')
bts_tr = pd.read_csv(DATA / 'BioclimTimeSeries/values/GLC25-PA-train-bioclimatic_monthly.csv')
bts_te = pd.read_csv(DATA / 'BioclimTimeSeries/values/GLC25-PA-test-bioclimatic_monthly.csv')

def compute_ts_stats(df):
    """Pour chaque variable (pr, tas, tasmax, tasmin), calcule mean/std/min/max/trend"""
    sid = df[['surveyId']].copy()
    feat_cols = [c for c in df.columns if c != 'surveyId']
    vals = df[feat_cols]

    # Séparer par variable
    stats = [sid]
    for var in ['pr', 'tas', 'tasmax', 'tasmin']:
        cols = [c for c in feat_cols if f'Bio-{var}_' in c]
        sub = vals[cols]
        stats.append(sub.mean(axis=1).rename(f'{var}_mean'))
        stats.append(sub.std(axis=1).rename(f'{var}_std'))
        stats.append(sub.min(axis=1).rename(f'{var}_min'))
        stats.append(sub.max(axis=1).rename(f'{var}_max'))
        # Tendance : corrélation avec le temps (proxy de trend)
        time_idx = np.arange(len(cols))
        trend = sub.apply(lambda row: np.polyfit(time_idx, row.fillna(row.mean()), 1)[0], axis=1)
        stats.append(trend.rename(f'{var}_trend'))

    return pd.concat(stats, axis=1)

print('Calcul stats train...')
bts_stats_tr = compute_ts_stats(bts_tr)
print('Calcul stats test...')
bts_stats_te = compute_ts_stats(bts_te)
print(f'BioclimTS stats shape: {bts_stats_tr.shape}')

In [ ]:
# ─── 4. Merge de toutes les features ──────────────────────────────────────────
def merge_all(surveys, dfs):
    result = surveys.copy()
    for df in dfs:
        result = result.merge(df, on='surveyId', how='left')
    return result

all_dfs_tr = [elev_tr, soil_tr, land_tr, foot_tr, bioclim_tr, bts_stats_tr]
all_dfs_te = [elev_te, soil_te, land_te, foot_te, bioclim_te, bts_stats_te]

X_train_df = merge_all(train_surveys, all_dfs_tr)
X_test_df  = merge_all(test_surveys,  all_dfs_te)

# Encoder la région
reg_tr = pd.get_dummies(X_train_df['region'], prefix='region')
reg_te = pd.get_dummies(X_test_df['region'],  prefix='region').reindex(columns=reg_tr.columns, fill_value=0)
X_train_df = pd.concat([X_train_df.drop(columns=['region']), reg_tr], axis=1)
X_test_df  = pd.concat([X_test_df.drop(columns=['region']),  reg_te], axis=1)

train_df     = X_train_df.merge(train_labels, on='surveyId', how='inner')
feature_cols = [c for c in X_train_df.columns if c != 'surveyId']

X_train = train_df[feature_cols].values.astype(np.float32)
X_test  = X_test_df[feature_cols].values.astype(np.float32)

# Imputer NaN
col_medians = np.nanmedian(X_train, axis=0)
for i in range(X_train.shape[1]):
    X_train[np.isnan(X_train[:, i]), i] = col_medians[i]
    X_test[np.isnan(X_test[:, i]), i]   = col_medians[i]

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Baseline avait 75 features, maintenant : {X_train.shape[1]} features')

In [ ]:
# ─── 5. Encodage multi-label ──────────────────────────────────────────────────
all_species = sorted(pa_train['speciesId'].dropna().astype(int).unique())
mlb = MultiLabelBinarizer(classes=all_species)
Y_train = mlb.fit_transform(train_df['species_list'])
print(f'Y_train: {Y_train.shape}')

In [ ]:
# ─── 6. Entraînement XGBoost GPU ──────────────────────────────────────────────
species_counts = pa_train['speciesId'].dropna().astype(int).value_counts()
MIN_OCC = 10
common_species = species_counts[species_counts >= MIN_OCC].index.tolist()
species_to_idx = {s: i for i, s in enumerate(mlb.classes_)}
common_idx = [species_to_idx[s] for s in common_species if s in species_to_idx]
print(f'Espèces à entraîner: {len(common_species)}')

xgb_params = dict(
    objective='binary:logistic',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    eval_metric='logloss',
    verbosity=0,
)

test_probs = np.zeros((len(X_test), len(all_species)), dtype=np.float32)

for i, (sp_idx, sp_id) in enumerate(zip(common_idx, common_species)):
    y = Y_train[:, sp_idx]
    if y.sum() < MIN_OCC:
        continue
    clf = XGBClassifier(**xgb_params)
    clf.fit(X_train, y)
    test_probs[:, sp_idx] = clf.predict_proba(X_test)[:, 1]
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(common_idx)} espèces...')

print('Entraînement terminé!')

In [ ]:
# ─── 7. Génération soumission ─────────────────────────────────────────────────
THRESHOLD = 0.2
TOP_K = 10

predictions = []
for i in range(len(X_test)):
    probs = test_probs[i]
    above = np.where(probs >= THRESHOLD)[0]
    if len(above) < TOP_K:
        top_k = np.argsort(probs)[::-1][:TOP_K]
        pred_idx = np.union1d(above, top_k)
    else:
        pred_idx = above
    pred_species = sorted([int(mlb.classes_[j]) for j in pred_idx])
    predictions.append(' '.join(map(str, pred_species)))

submission = pd.DataFrame({
    'surveyId': X_test_df['surveyId'].values,
    'predictions': predictions
})
submission.to_csv('/kaggle/working/submission_xgb_ts.csv', index=False)
print(f'Soumission sauvegardée! {len(submission)} surveys')
submission.head()